# NYC Weather Data from Open-Meteo API

## Import Additional Packages

In [0]:
import requests
import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    DoubleType,
    FloatType,
)

In [0]:
from pyspark import pipelines as dp

## Weather API Configuration

In [0]:
# Open-Meteo API configuration for NYC area
API_URL = "https://archive-api.open-meteo.com/v1/archive"
LATITUDE = 41
LONGITUDE = -74
START_DATE = "2025-01-01"
END_DATE = "2025-12-31"
TIMEZONE = "America/New_York"

## Create Weather Schema

In [0]:
weather_schema = StructType(
    [
        StructField(
            name="observation_time",
            dataType=TimestampType(),
            nullable=False,
            metadata={"comment": "Timestamp of weather observation (hourly)"}
        ),
        StructField(
            name="temperature_2m",
            dataType=FloatType(),
            nullable=True,
            metadata={"comment": "Temperature at 2 meters above ground in °C"}
        ),
        StructField(
            name="relative_humidity_2m",
            dataType=FloatType(),
            nullable=True,
            metadata={"comment": "Relative humidity at 2 meters above ground in %"}
        ),
        StructField(
            name="apparent_temperature",
            dataType=FloatType(),
            nullable=True,
            metadata={"comment": "Apparent temperature (feels like) in °C"}
        ),
        StructField(
            name="precipitation",
            dataType=FloatType(),
            nullable=True,
            metadata={"comment": "Total precipitation in mm"}
        ),
        StructField(
            name="wind_speed_10m",
            dataType=FloatType(),
            nullable=True,
            metadata={"comment": "Wind speed at 10 meters above ground in km/h"}
        ),
        StructField(
            name="latitude",
            dataType=DoubleType(),
            nullable=False,
            metadata={"comment": "Latitude of observation point"}
        ),
        StructField(
            name="longitude",
            dataType=DoubleType(),
            nullable=False,
            metadata={"comment": "Longitude of observation point"}
        ),
        StructField(
            name="sys_Insert_Dt",
            dataType=TimestampType(),
            nullable=False,
            metadata={"comment": "Timestamp when record was loaded"}
        ),
        StructField(
            name="sys_Insert_Fp",
            dataType=StringType(),
            nullable=False,
            metadata={"comment": "Source of the data"}
        ),
    ]
)

## Weather ETL Pipeline

In [0]:
@dp.temporary_view(name="dwh_nyc_weather_basis")
def bronze_dwh_nyc_weather_basis():
    """
    Fetch hourly weather data from Open-Meteo API and transform to DataFrame.
    Returns a DataFrame with hourly weather observations for NYC area.
    """
    # Build API URL with parameters
    params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "hourly": "temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,wind_speed_10m",
        "timezone": TIMEZONE
    }
    
    # Fetch data from API
    response = requests.get(API_URL, params=params)
    response.raise_for_status()  # Raise exception for bad status codes
    data = response.json()
    
    # Extract hourly data (columnar format)
    hourly = data['hourly']
    latitude = data['latitude']
    longitude = data['longitude']
    
    # Transform columnar data to rows
    rows = []
    for i in range(len(hourly['time'])):
        rows.append({
            'observation_time': hourly['time'][i],
            'temperature_2m': hourly.get('temperature_2m', [None] * len(hourly['time']))[i],
            'relative_humidity_2m': hourly.get('relative_humidity_2m', [None] * len(hourly['time']))[i],
            'apparent_temperature': hourly.get('apparent_temperature', [None] * len(hourly['time']))[i],
            'precipitation': hourly.get('precipitation', [None] * len(hourly['time']))[i],
            'wind_speed_10m': hourly.get('wind_speed_10m', [None] * len(hourly['time']))[i],
            'latitude': latitude,
            'longitude': longitude,
        })
    
    # Create DataFrame from rows
    df = spark.createDataFrame(rows)
    
    # Add system columns
    df = df.withColumn("sys_Insert_Dt", F.current_timestamp())
    df = df.withColumn("sys_Insert_Fp", F.lit(API_URL))
    
    # Convert observation_time string to timestamp
    df = df.withColumn("observation_time", F.to_timestamp(F.col("observation_time"), "yyyy-MM-dd'T'HH:mm"))
    
    # Fit DataFrame to schema
    schema_columns = [(field.name, field.dataType) for field in weather_schema.fields]
    df = df.select([F.col(col_name).cast(col_dtype) for col_name, col_dtype in schema_columns])
    
    return df

# Create streaming table for weather data
dp.create_streaming_table(
    "analytics.bronze.dwh_nyc_weather",
    comment="Hourly weather data for NYC area from Open-Meteo API (2025)",
    schema=weather_schema
)

# Apply CDC flow to maintain current state
dp.create_auto_cdc_from_snapshot_flow(
    target="analytics.bronze.dwh_nyc_weather",
    source="dwh_nyc_weather_basis",
    keys=["observation_time", "latitude", "longitude"],
    stored_as_scd_type=1,
)